## ANN Architecture

In [30]:
import numpy as np


def logistic(values):
    values = np.asarray(values, dtype=float)
    return np.exp(-np.logaddexp(0.0, -values))


ACTIVATIONS = {
    'logistic': logistic,
    'sigmoid': lambda values: 1 / (1 + np.exp(-values)),
    'relu': lambda values: np.maximum(0.0, values),
    'tanh': np.tanh,
    'linear': lambda values: values,
}


class FeedForwardANN:
    """Dense feedforward architecture; PSO candidates hold its trainable parameters."""

    def __init__(self, layer_sizes, activations):
        self.layer_sizes = tuple(layer_sizes)
        self.activations = tuple(activations)

        self.layout = []
        offset = 0
        for fan_in, fan_out in zip(self.layer_sizes[:-1], self.layer_sizes[1:]):
            weight_end = offset + fan_in * fan_out # End index of the weight matrix for this layer in the flat vector
            bias_end = weight_end + fan_out # End index of the bias vector for this layer in the flat vector
            self.layout.append((offset, weight_end, bias_end, fan_in, fan_out))
            offset = bias_end # Update offset to the end of the bias vector for the next layer
        
        self.n_parameters = offset
        
        self.weight_indices = np.concatenate([
            np.arange(start, weight_end)
            for start, weight_end, _, _, _ in self.layout
        ])

    def unpack(self, particles):
        """View flat particle vectors as per-layer weight and bias tensors."""
        particles = np.asarray(particles, dtype=float)
        particles = np.atleast_2d(particles)
        if particles.ndim != 2 or particles.shape[1] != self.n_parameters:
            raise ValueError(f'Expected particles with {self.n_parameters} parameters.')
        
        for start, weight_end, bias_end, fan_in, fan_out in self.layout:
            yield (
                particles[:, start:weight_end].reshape(-1, fan_in, fan_out),
                particles[:, weight_end:bias_end, None].transpose(0, 2, 1),
            )

    def forward(self, features, particles, return_logits=False):
        """Evaluate one or many particle-defined networks on a feature matrix."""
        features = np.asarray(features, dtype=float)
        outputs = features[None, :, :]
        
        for layer_index, ((weights, biases), name) in enumerate(
            zip(self.unpack(particles), self.activations)
        ):
            logits = outputs @ weights + biases
            if return_logits and layer_index == len(self.activations) - 1:
                return logits
            outputs = ACTIVATIONS[name](logits)
            
        return outputs

    def predict_proba(self, features, parameters):
        """Return class-1 probabilities for a single trained binary classifier."""
        return self.forward(features, parameters)[0, :, 0]

    def initialize(self, n_particles, rng):
        """Create a fresh swarm using activation-aware random weight scales."""
        particles = np.empty((n_particles, self.n_parameters))
        for (start, weight_end, bias_end, fan_in, fan_out), name in zip(
            self.layout, self.activations
        ):
            limit = np.sqrt(6.0 / (fan_in if name == 'relu' else fan_in + fan_out))
            particles[:, start:weight_end] = rng.uniform(
                -limit, limit, (n_particles, fan_in * fan_out)
            )
            particles[:, weight_end:bias_end] = rng.normal(0.0, 0.1, (n_particles, fan_out))
        return particles


def binary_losses(network, features, labels, particles, l2=0.0):
    """Compute each particle's mean logit-based BCE plus weight-only L2 penalty."""
    labels = np.asarray(labels)

    # The logits form avoids taking log of rounded probabilities near 0 or 1.
    logits = network.forward(features, particles, return_logits=True)[:, :, 0]
    losses = np.mean(np.logaddexp(0.0, logits) - labels[None, :] * logits, axis=1)
    weights = np.atleast_2d(particles)[:, network.weight_indices]
    return losses + l2 * np.mean(weights ** 2, axis=1)

## PSO Optimisation

# PSO Questions - What are we improving?

### Inside each PSO run
PSO optimizes **all ANN weights and biases**. Each particle’s position is one complete set of network parameters.

These are **trainable parameters, not hyperparameters**. The objective being minimized is:

**Mean binary cross-entropy + weight-only L2 regularization.**

### Across experiments
The notebook compares architectures and predefined hyperparameter profiles using cross-validation:

| Hyperparameter | Purpose | Values compared |
|---|---|---|
| `particles` | Swarm size | 32, 48 |
| `iterations` | Search duration | 180, 350 |
| `c1` | Attraction to personal best | 1.49618, 1.7, 1.3 |
| `c2` | Attraction to global best | 1.49618, 1.3, 1.7 |
| `informant_count` | Informants per particle, including itself | 3, 5, 7 |
| `inertia` | Starting/ending momentum | (0.7298, 0.4), (0.9, 0.4), (0.8, 0.35) |
| `bound` | Weight/bias magnitude limit | 6, 8, 4 |
| `velocity_fraction` | Maximum movement relative to search width | 0.15, 0.10 |
| `l2` | Regularization strength | 1e-4, 1e-5, 1e-3 |

In [31]:
class BinaryCrossEntropyObjective:
    def __init__(self, network, features, labels, l2=0.0):
        self.network = network
        self.features = np.asarray(features, dtype=float)
        self.labels = np.asarray(labels)
        self.l2 = float(l2)

    def __call__(self, particles):
        return binary_losses(
            self.network, self.features, self.labels, particles, l2=self.l2
        )


class ParticleSwarmOptimizer:
    def __init__(self, objective, seed=0, iterations=250, c1=1.49618,
                 c2=1.49618, inertia=(0.7298, 0.4), bound=6.0,
                 velocity_fraction=0.15, *, c_informant=0.0, informant_count=3):
        self.objective = objective
        self.seed = seed
        self.iterations = iterations
        self.c1 = c1
        self.c2 = c2
        self.c_informant = c_informant
        self.informant_count = informant_count
        self.inertia = tuple(inertia)
        self.bound = bound
        self.velocity_fraction = velocity_fraction
        self.result = None
        controls = np.asarray(
            [c1, c2, c_informant, bound, velocity_fraction, *self.inertia], dtype=float
        )

    def _informant_bests(self, personal_positions, personal_costs, rng):
        """Resample each particle's informants and return copies of their best positions."""
        swarm_size = len(personal_positions)

        best_indices = np.empty(swarm_size, dtype=int)
        for particle_index in range(swarm_size):
            others = np.delete(np.arange(swarm_size), particle_index)
            selected = rng.choice(others, size=self.informant_count - 1, replace=False)
            informants = np.concatenate(([particle_index], selected))
            best_indices[particle_index] = informants[np.argmin(personal_costs[informants])]
        return personal_positions[best_indices].copy()

    def _evaluate(self, positions):
        """Require one finite objective value for each particle position."""
        costs = np.asarray(self.objective(positions), dtype=float)
        return costs

    def optimize(self, initial_positions):
        """Search from a swarm matrix and return its best candidate and history."""
        positions = np.asarray(initial_positions, dtype=float).copy()
        rng = np.random.default_rng(self.seed)
        positions = np.clip(positions, -self.bound, self.bound)
        velocity_limit = 2.0 * self.bound * self.velocity_fraction
        velocities = rng.uniform(-0.1, 0.1, positions.shape) * velocity_limit

        # Keep independent snapshots: later particle movement must not mutate bests.
        personal_positions = positions.copy()
        personal_costs = self._evaluate(positions)
        best_index = int(np.argmin(personal_costs))
        global_position = personal_positions[best_index].copy()
        global_cost = float(personal_costs[best_index])
        history = [global_cost]

        for iteration in range(self.iterations):
            progress = iteration / max(self.iterations - 1, 1)
            momentum = self.inertia[0] + progress * (self.inertia[1] - self.inertia[0])
            cognitive_random = rng.random(positions.shape)
            social_random = rng.random(positions.shape)

            velocities = (
                momentum * velocities
                + self.c1 * cognitive_random * (personal_positions - positions)
                + self.c2 * social_random * (global_position - positions)
            )
            if self.c_informant > 0:
                informant_positions = self._informant_bests(personal_positions, personal_costs, rng)
                informant_random = rng.random(positions.shape)
                velocities += self.c_informant * informant_random * (informant_positions - positions)
                
            velocities = np.clip(velocities, -velocity_limit, velocity_limit)
            proposed_positions = positions + velocities
            clipped = (proposed_positions < -self.bound) | (proposed_positions > self.bound)
            positions = np.clip(proposed_positions, -self.bound, self.bound)
            velocities[clipped] = 0.0

            # Update personal and global bests only after evaluating the whole swarm.
            costs = self._evaluate(positions)
            improved = costs < personal_costs
            personal_positions[improved] = positions[improved].copy()
            personal_costs[improved] = costs[improved]
            best_index = int(np.argmin(personal_costs))
            if personal_costs[best_index] < global_cost:
                global_position = personal_positions[best_index].copy()
                global_cost = float(personal_costs[best_index])
                
            history.append(global_cost)

        self.result = {
            'parameters': global_position,
            'cost': global_cost,
            'history': np.asarray(history),
            'evaluations': len(positions) * (self.iterations + 1),
        }
        return self.result


class ANNTrainer:
    """Connect an ANN architecture, its data objective, and a PSO run."""

    def __init__(self, network, config):
        self.network = network
        self.config = dict(config)

    def fit(self, features, labels, seed):
        # Every swarm starts from fresh candidate networks; only PSO learns weights.
        initial_positions = self.network.initialize(
            self.config['particles'], np.random.default_rng(seed)
        )
        objective = BinaryCrossEntropyObjective(
            self.network, features, labels, l2=self.config['l2']
        )
        optimizer = ParticleSwarmOptimizer(
            objective,
            seed=seed + 10000,
            iterations=self.config['iterations'],
            c1=self.config['c1'],
            c2=self.config['c2'],
            c_informant=self.config.get('c_informant', 0.0),
            informant_count=self.config.get('informant_count', 3),
            inertia=self.config['inertia'],
            bound=self.config['bound'],
            velocity_fraction=self.config['velocity_fraction'],
        )
        return optimizer.optimize(initial_positions)

In [ ]:
from pathlib import Path
import hashlib


DATA_OVERRIDE = None
DATA_RELATIVE = Path('outline/data_banknote_authentication.txt')
possible_roots = [Path.cwd(), *Path.cwd().parents]
possible_paths = [
    root / prefix / DATA_RELATIVE
    for root in possible_roots
    for prefix in (Path('.'), Path('Biologically-Inspired-Computation-Coursework'))
]
DATA_PATH = Path(DATA_OVERRIDE).expanduser() if DATA_OVERRIDE else next(
    (path for path in possible_paths if path.is_file()), None
)


parsed_rows = []
repaired_label_lines = []
for line_number, line in enumerate(DATA_PATH.read_text().splitlines(), start=1):
    if not line.strip():
        continue
    fields = line.split(',')

    target = fields[-1].strip()
    if target.endswith('\u00a7') and target[:-1].strip() in {'0', '1'}:
        fields[-1] = target[:-1].strip()
        repaired_label_lines.append(line_number)
    parsed_rows.append([float(field.strip()) for field in fields])

raw_data = np.asarray(parsed_rows, dtype=float)
if repaired_label_lines:
    print('Warning: removed a stray section sign from target label(s) in memory on line(s):',
          repaired_label_lines)

# Reject conflicting labels for duplicate feature vectors before removing exact rows.
unique_features, feature_groups = np.unique(raw_data[:, :4], axis=0, return_inverse=True)
_, first_indices = np.unique(raw_data, axis=0, return_index=True)
retained_indices = np.sort(first_indices)
data = raw_data[retained_indices]
features = data[:, :4]
labels = data[:, -1].astype(int)
DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
FEATURE_NAMES = ('variance', 'skewness', 'curtosis', 'entropy')


def stratified_holdout(targets, fraction=0.2, seed=2026):
    rng = np.random.default_rng(seed)
    development, held_out = [], []
    for label in np.unique(targets):
        indices = rng.permutation(np.flatnonzero(targets == label))
        cut = max(1, int(round(len(indices) * fraction)))
        held_out.extend(indices[:cut])
        development.extend(indices[cut:])
    return (np.asarray(rng.permutation(development), dtype=int),
            np.asarray(rng.permutation(held_out), dtype=int))


def stratified_folds(targets, count=3, seed=31415):
    rng = np.random.default_rng(seed)
    buckets = [[] for _ in range(count)]
    for label in np.unique(targets):
        shuffled = rng.permutation(np.flatnonzero(targets == label))
        for bucket, chunk in zip(buckets, np.array_split(shuffled, count)):
            bucket.extend(chunk.tolist())
    all_indices = np.arange(len(targets))
    return [(np.setdiff1d(all_indices, validation), np.asarray(validation, dtype=int))
            for validation in buckets]


def fit_scaler(training_features):
    mean = np.mean(training_features, axis=0)
    scale = np.std(training_features, axis=0)
    return mean, np.where(scale > 0, scale, 1.0)


DEVELOPMENT_INDICES, TEST_INDICES = stratified_holdout(labels)
DEVELOPMENT_INDICES = np.asarray(DEVELOPMENT_INDICES, dtype=int)
TEST_INDICES = np.asarray(TEST_INDICES, dtype=int)
features_development = features[DEVELOPMENT_INDICES]
labels_development = labels[DEVELOPMENT_INDICES]
CV_FOLDS = stratified_folds(labels_development)
FOLD_DATA = []
for training_indices, validation_indices in CV_FOLDS:
    mean, scale = fit_scaler(features_development[training_indices])
    FOLD_DATA.append((
        (features_development[training_indices] - mean) / scale,
        labels_development[training_indices],
        (features_development[validation_indices] - mean) / scale,
        labels_development[validation_indices],
    ))


def class_balance_row(split_name, targets):
    targets = np.asarray(targets)
    counts = np.bincount(targets.astype(int), minlength=2)
    return {
        'split': split_name, 'samples': len(targets),
        'class_0': int(counts[0]), 'class_1': int(counts[1]),
        'class_1_fraction': float(counts[1] / len(targets)),
        'majority_minority_ratio': float(counts.max() / counts.min()),
    }


CLASS_BALANCE_ROWS = [
    class_balance_row('Unique data', labels),
    class_balance_row('Development', labels_development),
    class_balance_row('Locked test (descriptive only)', labels[TEST_INDICES]),
]

FOLD_BASELINE_ROWS = []
for fold_index, (training_indices, validation_indices) in enumerate(CV_FOLDS, 1):
    training_targets = labels_development[training_indices]
    validation_targets = labels_development[validation_indices]
    CLASS_BALANCE_ROWS.extend([
        class_balance_row(f'Fold {fold_index} training', training_targets),
        class_balance_row(f'Fold {fold_index} validation', validation_targets),
    ])
    fold_majority = int(np.argmax(np.bincount(training_targets, minlength=2)))
    FOLD_BASELINE_ROWS.append({
        'fold': fold_index, 'training_majority_class': fold_majority,
        'validation_accuracy': float(np.mean(validation_targets == fold_majority)),
        'validation_balanced_accuracy': 0.5,
    })
    
print('\nClass balance before training (the holdout is not used to choose a remedy):')
for row in CLASS_BALANCE_ROWS:
    print(f"{row['split']}: n0={row['class_0']}, n1={row['class_1']}, "
          f"class 1={row['class_1_fraction']:.2%}, ratio={row['majority_minority_ratio']:.3f}:1")
print('Fold majority baselines (majority chosen using fold-training labels only):')
for row in FOLD_BASELINE_ROWS:
    print(row)


Class balance before training (the holdout is not used to choose a remedy):
Unique data: n0=738, n1=610, class 1=45.25%, ratio=1.210:1
Development: n0=590, n1=488, class 1=45.27%, ratio=1.209:1
Locked test (descriptive only): n0=148, n1=122, class 1=45.19%, ratio=1.213:1
Fold 1 training: n0=393, n1=325, class 1=45.26%, ratio=1.209:1
Fold 1 validation: n0=197, n1=163, class 1=45.28%, ratio=1.209:1
Fold 2 training: n0=393, n1=325, class 1=45.26%, ratio=1.209:1
Fold 2 validation: n0=197, n1=163, class 1=45.28%, ratio=1.209:1
Fold 3 training: n0=394, n1=326, class 1=45.28%, ratio=1.209:1
Fold 3 validation: n0=196, n1=162, class 1=45.25%, ratio=1.210:1
Fold majority baselines (majority chosen using fold-training labels only):
{'fold': 1, 'training_majority_class': 0, 'validation_accuracy': 0.5472222222222223, 'validation_balanced_accuracy': 0.5}
{'fold': 2, 'training_majority_class': 0, 'validation_accuracy': 0.5472222222222223, 'validation_balanced_accuracy': 0.5}
{'fold': 3, 'training_ma

In [33]:
from time import perf_counter


SEARCH_SEEDS = (11, 29)
BASE_PSO = {
    'particles': 32,
    'iterations': 180,
    'c1': 1.49618,
    'c2': 1.49618,
    'c_informant': 0.5,
    'informant_count': 3,
    'inertia': (0.7298, 0.4),
    'bound': 6.0,
    'velocity_fraction': 0.15,
    'l2': 1e-4,
}
ARCHITECTURES = [
    {'hidden': hidden, 'activations': (activation,) * len(hidden) + ('logistic',)}
    for hidden in ((4,), (8,), (12,), (8, 4))
    for activation in ('tanh', 'relu', 'logistic')
] + [
    {'hidden': (8, 4), 'activations': ('tanh', 'relu', 'logistic')},
    {'hidden': (8, 4), 'activations': ('relu', 'tanh', 'logistic')},
]


def make_network(config):
    """Build the ANN architecture described by one search candidate."""
    return FeedForwardANN(
        (features_development.shape[1], *config['hidden'], 1),
        config['activations'],
    )


def evaluate_config(config, fold_data=FOLD_DATA, seeds=SEARCH_SEEDS):
    """Evaluate one ANN/PSO configuration without consulting the test split."""
    network = make_network(config)
    trainer = ANNTrainer(network, config)
    scores = []
    start = perf_counter()
    for fold_index, (train_features, train_labels, valid_features, valid_labels) in enumerate(fold_data):
        for seed in seeds:
            # Each independent run gets a new swarm; validation only scores its final best ANN.
            fit = trainer.fit(train_features, train_labels, seed + 100 * fold_index)
            probabilities = network.predict_proba(valid_features, fit['parameters'])
            validation_bce = float(binary_losses(
                network, valid_features, valid_labels, fit['parameters'], l2=0.0
            )[0])
            scores.append({
                'fold': fold_index + 1,
                'seed': seed,
                'accuracy': float(np.mean((probabilities >= 0.5) == valid_labels)),
                'bce': validation_bce,
                'training_cost': fit['cost'],
                'evaluations': fit['evaluations'],
            })
    accuracies = np.array([score['accuracy'] for score in scores])
    return {
        'config': dict(config),
        'parameters': network.n_parameters,
        'mean_accuracy': float(accuracies.mean()),
        'std_accuracy': float(accuracies.std(ddof=1)) if len(scores) > 1 else 0.0,
        'mean_bce': float(np.mean([score['bce'] for score in scores])),
        'scores': scores,
        'seconds': perf_counter() - start,
    }


def ranking_key(result):
    return (-result['mean_accuracy'], result['mean_bce'], result['parameters'])


def print_leaderboard(results, limit=10):
    print('rank | hidden | activations | CV accuracy +/- SD | BCE | parameters | seconds')
    for rank, result in enumerate(sorted(results, key=ranking_key)[:limit], 1):
        config = result['config']
        print(f"{rank:>4} | {str(config['hidden']):<8} | {str(config['activations']):<36} | "
              f"{result['mean_accuracy']:.4%} +/- {result['std_accuracy']:.4%} | "
              f"{result['mean_bce']:.5f} | {result['parameters']:>4} | {result['seconds']:.1f}")

In [34]:
architecture_results = []
for candidate_number, architecture in enumerate(ARCHITECTURES, 1):
    result = evaluate_config(dict(BASE_PSO, **architecture))
    architecture_results.append(result)
    print(f"Architecture {candidate_number:02}/{len(ARCHITECTURES)}: "
          f"{architecture['hidden']} {architecture['activations']} | "
          f"CV {result['mean_accuracy']:.4%} | {result['seconds']:.1f}s", flush=True)
print_leaderboard(architecture_results)


TOP_ARCHITECTURES = 3
REFINEMENT_PSO = [
    dict(BASE_PSO, particles=48, iterations=350, informant_count=3),
    dict(BASE_PSO, particles=48, iterations=350, c1=1.7, c2=1.3, informant_count=5,
         inertia=(0.9, 0.4), bound=8.0, l2=1e-5),
    dict(BASE_PSO, particles=48, iterations=350, c1=1.3, c2=1.7, informant_count=7,
         inertia=(0.8, 0.35), bound=4.0, velocity_fraction=0.1, l2=1e-3),
]
refinement_results = []
shortlist = sorted(architecture_results, key=ranking_key)[:TOP_ARCHITECTURES]
for architecture_rank, architecture_result in enumerate(shortlist, 1):
    for profile_number, pso_settings in enumerate(REFINEMENT_PSO, 1):
        previous_config = architecture_result['config']
        config = dict(pso_settings, hidden=previous_config['hidden'],
                      activations=previous_config['activations'])
        result = evaluate_config(config)
        refinement_results.append(result)
        print(f"Refinement {architecture_rank}/{TOP_ARCHITECTURES}, profile {profile_number}/3: "
              f"CV {result['mean_accuracy']:.4%} | {result['seconds']:.1f}s", flush=True)
print_leaderboard(refinement_results, limit=len(refinement_results))

best_result = min(architecture_results + refinement_results, key=ranking_key)
BEST_CONFIG = dict(best_result['config'])
assert all(np.isfinite(result['mean_bce']) for result in architecture_results + refinement_results)
print('\nBest configuration among the evaluated candidates:')
print(BEST_CONFIG)
print(f"Validation accuracy: {best_result['mean_accuracy']:.4%} +/- "
      f"{best_result['std_accuracy']:.4%} (fold/seed SD, not a confidence interval)")
print('The locked test set has not been evaluated.')

Architecture 01/14: (4,) ('tanh', 'logistic') | CV 99.9537% | 1.4s
Architecture 02/14: (4,) ('relu', 'logistic') | CV 100.0000% | 1.1s
Architecture 03/14: (4,) ('logistic', 'logistic') | CV 99.8609% | 2.6s
Architecture 04/14: (8,) ('tanh', 'logistic') | CV 100.0000% | 1.6s
Architecture 05/14: (8,) ('relu', 'logistic') | CV 100.0000% | 1.0s
Architecture 06/14: (8,) ('logistic', 'logistic') | CV 99.9537% | 4.3s
Architecture 07/14: (12,) ('tanh', 'logistic') | CV 99.9074% | 2.2s
Architecture 08/14: (12,) ('relu', 'logistic') | CV 100.0000% | 1.3s
Architecture 09/14: (12,) ('logistic', 'logistic') | CV 100.0000% | 6.2s
Architecture 10/14: (8, 4) ('tanh', 'tanh', 'logistic') | CV 99.9074% | 2.4s
Architecture 11/14: (8, 4) ('relu', 'relu', 'logistic') | CV 100.0000% | 1.4s
Architecture 12/14: (8, 4) ('logistic', 'logistic', 'logistic') | CV 99.9074% | 6.1s
Architecture 13/14: (8, 4) ('tanh', 'relu', 'logistic') | CV 99.9537% | 2.1s
Architecture 14/14: (8, 4) ('relu', 'tanh', 'logistic') | CV

## Controlled parameter-bound sensitivity

Compare position bounds 4, 6 and 8 for the selected architecture and PSO profile, changing no other settings except `velocity_fraction` to keep the absolute velocity cap fixed at the baseline value of 1.8. Each bound uses the same development folds and search seeds; no locked-test rows are used and no model artifacts are written. Because the architecture/profile was selected using these same folds, treat this as a controlled sensitivity check, not an independent estimate of generalization or proof that one bound is universally best.

In [35]:
BOUND_SENSITIVITY_VALUES = (4.0, 6.0, 8.0)
BOUND_SENSITIVITY_VELOCITY_LIMIT = (
    2.0 * BASE_PSO['bound'] * BASE_PSO['velocity_fraction']
)
BOUND_SENSITIVITY_RESULTS = {}

for position_bound in BOUND_SENSITIVITY_VALUES:
    velocity_fraction = BOUND_SENSITIVITY_VELOCITY_LIMIT / (2.0 * position_bound)
    config = dict(
        BEST_CONFIG,
        bound=position_bound,
        velocity_fraction=velocity_fraction,
    )
    actual_velocity_limit = 2.0 * config['bound'] * config['velocity_fraction']
    result = evaluate_config(config)
    BOUND_SENSITIVITY_RESULTS[position_bound] = result
    print(f"Bound {position_bound:g}: velocity_fraction={velocity_fraction:.4f}, "
          f"velocity cap=+/-{actual_velocity_limit:.2f}, "
          f"CV accuracy={result['mean_accuracy']:.4%}, "
          f"BCE={result['mean_bce']:.6f}, "
          f"evaluations/run={result['scores'][0]['evaluations']}, "
          f"time={result['seconds']:.1f}s", flush=True)

bound_reference = BOUND_SENSITIVITY_RESULTS[6.0]
print('\nPaired summary relative to bound=6 (same folds and seeds):')
print('bound | accuracy +/- SD | delta accuracy (pp) | mean BCE | delta BCE')
for position_bound in BOUND_SENSITIVITY_VALUES:
    result = BOUND_SENSITIVITY_RESULTS[position_bound]
    accuracy_delta = 100.0 * (result['mean_accuracy'] - bound_reference['mean_accuracy'])
    bce_delta = result['mean_bce'] - bound_reference['mean_bce']
    print(f"{position_bound:>5.1f} | {result['mean_accuracy']:.4%} +/- "
          f"{result['std_accuracy']:.4%} | {accuracy_delta:+.4f} | "
          f"{result['mean_bce']:.6f} | {bce_delta:+.6f}")

Bound 4: velocity_fraction=0.2250, velocity cap=+/-1.80, CV accuracy=100.0000%, BCE=0.000011, evaluations/run=16848, time=4.9s
Bound 6: velocity_fraction=0.1500, velocity cap=+/-1.80, CV accuracy=100.0000%, BCE=0.000071, evaluations/run=16848, time=4.4s
Bound 8: velocity_fraction=0.1125, velocity cap=+/-1.80, CV accuracy=100.0000%, BCE=0.000012, evaluations/run=16848, time=4.2s

Paired summary relative to bound=6 (same folds and seeds):
bound | accuracy +/- SD | delta accuracy (pp) | mean BCE | delta BCE
  4.0 | 100.0000% +/- 0.0000% | +0.0000 | 0.000011 | -0.000060
  6.0 | 100.0000% +/- 0.0000% | +0.0000 | 0.000071 | +0.000000
  8.0 | 100.0000% +/- 0.0000% | +0.0000 | 0.000012 | -0.000059


In [36]:
import json
import platform


FINAL_SEEDS = (101, 202, 303)
final_network = make_network(BEST_CONFIG)
final_mean, final_scale = fit_scaler(features_development)
scaled_development = (features_development - final_mean) / final_scale
final_trainer = ANNTrainer(final_network, BEST_CONFIG)
# Independent trainer runs differ only by their deterministic swarm seeds.
final_runs = [
    (seed, final_trainer.fit(scaled_development, labels_development, seed))
    for seed in FINAL_SEEDS
]
FINAL_SEED, final_fit = min(final_runs, key=lambda item: item[1]['cost'])
FINAL_PARAMETERS = final_fit['parameters'].copy()
print('Final restart selected by development training objective only:', FINAL_SEED)


def classification_report(targets, probabilities):
    targets = np.asarray(targets)
    probabilities = np.asarray(probabilities)
    if probabilities.shape != targets.shape or not np.isfinite(probabilities).all():
        raise ValueError('Provide one finite probability per target.')
    if not np.isin(targets, [0, 1]).all() or np.any((probabilities < 0) | (probabilities > 1)):
        raise ValueError('Expected binary labels and probabilities in [0, 1].')
    predictions = probabilities >= 0.5
    confusion = np.zeros((2, 2), dtype=int)
    np.add.at(confusion, (targets.astype(int), predictions.astype(int)), 1)
    true_negative, false_positive, false_negative, true_positive = confusion.ravel()
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    specificity = true_negative / max(true_negative + false_positive, 1)
    positives = probabilities[targets == 1]
    negatives = probabilities[targets == 0]
    auc = float(np.mean(
        (positives[:, None] > negatives[None, :]).astype(float)
        + 0.5 * (positives[:, None] == negatives[None, :])
    )) if len(positives) and len(negatives) else None
    accuracy = float(np.mean(predictions == targets))
    sample_size = len(targets)
    z_score = 1.959963984540054
    denominator = 1 + z_score ** 2 / sample_size
    center = (accuracy + z_score ** 2 / (2 * sample_size)) / denominator
    radius = z_score * np.sqrt(
        accuracy * (1 - accuracy) / sample_size + z_score ** 2 / (4 * sample_size ** 2)
    ) / denominator
    return {
        'samples': sample_size, 'accuracy': accuracy,
        'balanced_accuracy': float((recall + specificity) / 2),
        'precision_class_1': float(precision), 'recall_class_1': float(recall),
        'f1_class_1': float(2 * precision * recall / max(precision + recall, 1e-15)),
        'roc_auc': auc, 'confusion_matrix': confusion.tolist(),
        'accuracy_wilson_95': [float(center - radius), float(center + radius)],
    }


metric_probe = classification_report(np.array([0, 0, 1, 1]), np.array([0.1, 0.8, 0.2, 0.9]))
assert metric_probe['confusion_matrix'] == [[1, 1], [1, 1]]
assert metric_probe['accuracy'] == 0.5 and metric_probe['roc_auc'] == 0.75
assert classification_report(np.array([0, 1]), np.array([0.5, 0.5]))['roc_auc'] == 0.5

scaled_test = (features[TEST_INDICES] - final_mean) / final_scale
test_labels = labels[TEST_INDICES]
test_probabilities = final_network.predict_proba(scaled_test, FINAL_PARAMETERS)
TEST_REPORT = classification_report(test_labels, test_probabilities)
TEST_REPORT['binary_cross_entropy'] = float(binary_losses(
    final_network, scaled_test, test_labels, FINAL_PARAMETERS
)[0])
majority_class = int(np.argmax(np.bincount(labels_development)))
TEST_REPORT['majority_baseline_accuracy'] = float(np.mean(test_labels == majority_class))
assert np.isfinite(test_probabilities).all()
assert np.isclose(sum(sum(row) for row in TEST_REPORT['confusion_matrix']), len(TEST_INDICES))
print('Locked test results (threshold fixed at 0.5):')
print(json.dumps(TEST_REPORT, indent=2))
print('Confusion matrix: rows = actual class [0, 1], columns = predicted class [0, 1].')

ARTIFACT_DIR = DATA_PATH.parent.parent / 'Riddhi' / 'ann_results'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    ARTIFACT_DIR / 'best_model.npz', parameters=FINAL_PARAMETERS,
    layer_sizes=np.asarray(final_network.layer_sizes),
    activations=np.asarray(final_network.activations), mean=final_mean, scale=final_scale,
    feature_names=np.asarray(FEATURE_NAMES), threshold=np.asarray(0.5),
    development_indices=DEVELOPMENT_INDICES, test_indices=TEST_INDICES,
    retained_original_row_indices=retained_indices,
    final_training_history=final_fit['history'],
)
experiment = {
    'owner': 'Riddhi', 'source_notebook': 'ann.ipynb',
    'data_sha256': DATA_SHA256, 'python': platform.python_version(), 'numpy': np.__version__,
    'duplicate_policy': 'Remove exact duplicate rows before splitting; retain near duplicates.',
    'split_seed': 2026, 'fold_seed': 31415, 'fold_count': len(CV_FOLDS),
    'search_seeds': SEARCH_SEEDS, 'final_seeds': FINAL_SEEDS, 'selected_final_seed': FINAL_SEED,
    'ranking': 'Mean validation accuracy descending, BCE ascending, parameter count ascending.',
    'best_config': BEST_CONFIG,
    'selected_validation_accuracy': best_result['mean_accuracy'],
    'selected_validation_bce': best_result['mean_bce'],
    'architecture_results': architecture_results, 'refinement_results': refinement_results,
    'test_report': TEST_REPORT,
}
with (ARTIFACT_DIR / 'experiment.json').open('w') as stream:
    json.dump(experiment, stream, indent=2, allow_nan=False)
with (ARTIFACT_DIR / 'riddhi_results.json').open('w') as stream:
    json.dump({
        'owner': 'Riddhi', 'source': 'ann.ipynb',
        'protocol': 'Repeated stratified 3-fold CV; one locked final test holdout.',
        'experiment': experiment,
    }, stream, indent=2, allow_nan=False)
np.savetxt(
    ARTIFACT_DIR / 'test_predictions.csv',
    np.column_stack((retained_indices[TEST_INDICES], test_labels,
                     test_probabilities, (test_probabilities >= 0.5).astype(int))),
    delimiter=',', header='original_row_index,actual_class,probability_class_1,predicted_class',
    comments='', fmt=['%d', '%d', '%.17g', '%d'],
)
with np.load(ARTIFACT_DIR / 'best_model.npz', allow_pickle=False) as saved:
    restored_network = FeedForwardANN(saved['layer_sizes'].tolist(), saved['activations'].tolist())
    restored_probabilities = restored_network.predict_proba(
        (features[TEST_INDICES] - saved['mean']) / saved['scale'], saved['parameters']
    )
np.testing.assert_array_equal(restored_probabilities, test_probabilities)
print('Metric checks and model save/load round trip passed. Artifacts:', ARTIFACT_DIR)

Final restart selected by development training objective only: 101
Locked test results (threshold fixed at 0.5):
{
  "samples": 270,
  "accuracy": 1.0,
  "balanced_accuracy": 1.0,
  "precision_class_1": 1.0,
  "recall_class_1": 1.0,
  "f1_class_1": 1.0,
  "roc_auc": 1.0,
  "confusion_matrix": [
    [
      148,
      0
    ],
    [
      0,
      122
    ]
  ],
  "accuracy_wilson_95": [
    0.9859719604283536,
    1.0000000000000002
  ],
  "binary_cross_entropy": 8.478738723096129e-07,
  "majority_baseline_accuracy": 0.5481481481481482
}
Confusion matrix: rows = actual class [0, 1], columns = predicted class [0, 1].
Metric checks and model save/load round trip passed. Artifacts: /Users/bristi/Desktop/Projects/HWU/HWU - Biologically-Inspired Computation/Riddhi/ann_results


In [38]:
def predict_banknotes(raw_features, model_path=ARTIFACT_DIR / 'best_model.npz'):
    """Predict numeric classes from raw features in FEATURE_NAMES order."""
    raw_features = np.asarray(raw_features, dtype=float)
    if raw_features.ndim == 1:
        raw_features = raw_features[None, :]
    if raw_features.ndim != 2 or not np.isfinite(raw_features).all():
        raise ValueError('Provide a finite feature row or matrix.')
    with np.load(model_path, allow_pickle=False) as saved_model:
        network = FeedForwardANN(
            saved_model['layer_sizes'].tolist(), saved_model['activations'].tolist()
        )
        if raw_features.shape[1] != network.layer_sizes[0]:
            raise ValueError(f'Expected {network.layer_sizes[0]} features per row.')
        probabilities = network.predict_proba(
            (raw_features - saved_model['mean']) / saved_model['scale'],
            saved_model['parameters'],
        )
        classes = (probabilities >= float(saved_model['threshold'])).astype(int)
    return {'probability_class_1': probabilities, 'predicted_class': classes}


example_predictions = predict_banknotes(features_development[:3])
assert example_predictions['predicted_class'].shape == (3,)
assert predict_banknotes(features_development[0])['predicted_class'].shape == (1,)
for invalid_features in ([1, 2, 3], [1, np.nan, 3, 4]):
    try:
        predict_banknotes(invalid_features)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Invalid features were accepted: {invalid_features}')
    
print('Feature order:', FEATURE_NAMES)
print('Example predictions on three DEVELOPMENT rows, not unseen evaluation data:')
print(example_predictions)
print('Inference checks passed: batch, single-row and invalid input handling.')

Feature order: ('variance', 'skewness', 'curtosis', 'entropy')
Example predictions on three DEVELOPMENT rows, not unseen evaluation data:
{'probability_class_1': array([1.        , 0.99999997, 1.        ]), 'predicted_class': array([1, 1, 1])}
Inference checks passed: batch, single-row and invalid input handling.


## Loss Graphs using Plotly

In [39]:
import plotly.graph_objects as go
from IPython.display import display


probability_grid = np.linspace(0.001, 0.999, 400)
illustrative_labels = np.array([1, 0, 1, 0])
illustrative_probabilities = np.array([0.9, 0.8, 0.5, 0.1])
illustrative_losses = -(
    illustrative_labels * np.log(illustrative_probabilities)
    + (1 - illustrative_labels) * np.log1p(-illustrative_probabilities)
)
development_logits = final_network.forward(
    scaled_development, FINAL_PARAMETERS, return_logits=True
)[0, :, 0]
development_row_losses = (
    np.logaddexp(0.0, development_logits) - labels_development * development_logits
)
development_bce = float(development_row_losses.mean())
selected_weights = FINAL_PARAMETERS[final_network.weight_indices]
development_penalty = float(BEST_CONFIG['l2'] * np.mean(selected_weights ** 2))
objective_components = np.array([
    development_bce, development_penalty, development_bce + development_penalty
])
np.testing.assert_allclose(illustrative_losses, -np.log([0.9, 0.2, 0.5, 0.9]))
np.testing.assert_allclose(objective_components[-1], final_fit['cost'], rtol=1e-10, atol=1e-12)
for restart_seed, restart_fit in final_runs:
    assert np.isfinite(restart_fit['history']).all()
    assert np.all(restart_fit['history'] > 0)
    assert np.all(np.diff(restart_fit['history']) <= 0)


plot_colors = {'class 0': '#e76f51', 'class 1': '#167d8d', 'selected': '#5b4b8a'}

# BCE curves for a single observation, for either possible true class.
loss_curve = go.Figure()
loss_curve.add_trace(go.Scatter(
    x=probability_grid,
    y=-np.log1p(-probability_grid),
    mode='lines',
    name='True class = 0',
    line={'color': plot_colors['class 0'], 'width': 3},
    hovertemplate='P(class 1) = %{x:.3f}<br>BCE = %{y:.4f}<extra>True class 0</extra>',
))
loss_curve.add_trace(go.Scatter(
    x=probability_grid,
    y=-np.log(probability_grid),
    mode='lines',
    name='True class = 1',
    line={'color': plot_colors['class 1'], 'width': 3},
    hovertemplate='P(class 1) = %{x:.3f}<br>BCE = %{y:.4f}<extra>True class 1</extra>',
))
loss_curve.add_vline(x=0.5, line_dash='dot', line_color='#718096')
loss_curve.update_layout(
    title='Binary cross-entropy for one observation',
    xaxis_title='Predicted probability of class 1',
    yaxis_title='Binary cross-entropy (nats)',
    template='plotly_white',
    hovermode='x unified',
    legend_title_text='Actual class',
    margin={'l': 65, 'r': 25, 't': 70, 'b': 60},
)

# A small worked batch makes the per-observation losses and their mean visible.
worked_batch = go.Figure(go.Bar(
    x=[f'Row {index}' for index in range(1, len(illustrative_losses) + 1)],
    y=illustrative_losses,
    marker_color=[plot_colors['class 1'] if label else plot_colors['class 0']
                  for label in illustrative_labels],
    customdata=np.column_stack((illustrative_labels, illustrative_probabilities)),
    hovertemplate=(
        '%{x}<br>True class = %{customdata[0]}'
        '<br>P(class 1) = %{customdata[1]:.2f}'
        '<br>Individual BCE = %{y:.4f}<extra></extra>'
    ),
))
worked_batch.add_hline(
    y=float(illustrative_losses.mean()),
    line_dash='dash',
    line_color=plot_colors['selected'],
    annotation_text=f'Mean BCE = {illustrative_losses.mean():.3f}',
    annotation_position='top right',
)
worked_batch.update_layout(
    title='Worked batch: mean BCE is the average of individual losses',
    xaxis_title='Illustrative observation',
    yaxis_title='Individual BCE (nats)',
    template='plotly_white',
    showlegend=False,
    margin={'l': 65, 'r': 25, 't': 80, 'b': 60},
)

# Compare the three final training restarts; the chosen seed is marked in the legend.
restart_history = go.Figure()
for restart_seed, restart_fit in final_runs:
    is_selected = restart_seed == FINAL_SEED
    restart_history.add_trace(go.Scatter(
        x=np.arange(len(restart_fit['history'])),
        y=restart_fit['history'],
        mode='lines',
        name=f'Seed {restart_seed}' + (' (selected)' if is_selected else ''),
        line={
            'color': plot_colors['selected'] if is_selected else None,
            'width': 3 if is_selected else 2,
        },
        hovertemplate='Iteration %{x}<br>Objective = %{y:.5g}<extra>%{fullData.name}</extra>',
    ))
restart_history.update_layout(
    title='Final PSO restarts: best-so-far training objective',
    xaxis_title='PSO iteration (0 = initial swarm)',
    yaxis_title='Regularized training objective (log scale)',
    yaxis_type='log',
    template='plotly_white',
    hovermode='x unified',
    legend_title_text='Restart',
    margin={'l': 75, 'r': 25, 't': 70, 'b': 60},
)

# The waterfall shows how development BCE and the L2 term form the objective.
objective_breakdown = go.Figure(go.Waterfall(
    x=['Mean development BCE', 'Weight-only L2 penalty', 'Total objective'],
    measure=['relative', 'relative', 'total'],
    y=objective_components,
    text=[f'{value:.4g}' for value in objective_components],
    textposition='outside',
    connector={'line': {'color': '#a0aec0'}},
    increasing={'marker': {'color': plot_colors['class 1']}},
    totals={'marker': {'color': plot_colors['selected']}},
    hovertemplate='%{x}<br>Contribution = %{y:.6g}<extra></extra>',
))
objective_breakdown.update_layout(
    title='Selected model: development loss breakdown',
    xaxis_title='Total objective = mean BCE + weight-only L2 penalty',
    yaxis_title='Objective contribution',
    template='plotly_white',
    showlegend=False,
    margin={'l': 75, 'r': 25, 't': 70, 'b': 80},
)

loss_figures = [loss_curve, worked_batch, restart_history, objective_breakdown]
assert len(loss_figures) == 4
assert all(isinstance(figure, go.Figure) for figure in loss_figures)
for figure in loss_figures:
    display(figure)

print('Worked batch (illustrative data, not banknote samples):')
print('Row | True class | P(class 1) | Individual BCE')
for row_number, (label, probability, loss_value) in enumerate(
    zip(illustrative_labels, illustrative_probabilities, illustrative_losses), 1
):
    print(f'{row_number:>3} | {label:>10} | {probability:>10.2f} | {loss_value:.6f}')
print(f'Mean BCE = sum(individual losses) / 4 = {illustrative_losses.mean():.6f}')
print(f'Development objective = {development_bce:.8g} BCE + '
      f'{development_penalty:.8g} L2 = {objective_components[-1]:.8g}')
print('Plotly checks passed: worked loss calculation, objective decomposition and monotone PSO histories.')


Worked batch (illustrative data, not banknote samples):
Row | True class | P(class 1) | Individual BCE
  1 |          1 |       0.90 | 0.105361
  2 |          0 |       0.80 | 1.609438
  3 |          1 |       0.50 | 0.693147
  4 |          0 |       0.10 | 0.105361
Mean BCE = sum(individual losses) / 4 = 0.628327
Development objective = 4.7993348e-07 BCE + 1.76378e-05 L2 = 1.8117733e-05
Plotly checks passed: worked loss calculation, objective decomposition and monotone PSO histories.


## Checking why is the accuracy so good? Is it because of class imbalance?

In [40]:
import pandas as pd
import plotly.express as px


def imbalance_metrics(targets, probabilities):
    class_balance_row('Metric targets', targets)
    report = classification_report(targets, probabilities)
    confusion = np.asarray(report['confusion_matrix'])
    support = confusion.sum(axis=1)
    predicted_support = confusion.sum(axis=0)
    per_class = []
    for label in (0, 1):
        correct = int(confusion[label, label])
        per_class.append({
            'class': label, 'support': int(support[label]),
            'precision': float(correct / max(int(predicted_support[label]), 1)),
            'recall': float(correct / support[label]),
            'f1': float(2 * correct / (support[label] + predicted_support[label])),
        })
    return {
        'accuracy': report['accuracy'], 'balanced_accuracy': report['balanced_accuracy'],
        'macro_f1': float(np.mean([row['f1'] for row in per_class])),
        'brier_score': float(np.mean((np.asarray(probabilities) - np.asarray(targets)) ** 2)),
        'per_class': per_class,
    }


balance_frame = pd.DataFrame(CLASS_BALANCE_ROWS)
display(balance_frame)
balance_plot = balance_frame.melt(
    id_vars='split', value_vars=['class_0', 'class_1'],
    var_name='class', value_name='count',
)
px.bar(balance_plot, x='split', y='count', color='class', barmode='group',
       title='Class counts: check development and every fold before training').show()

development_majority = int(np.argmax(np.bincount(labels_development, minlength=2)))
diagnostic_results = {
    'Selected ANN (threshold 0.5)': imbalance_metrics(test_labels, test_probabilities),
    'Development-majority baseline': imbalance_metrics(
        test_labels, np.full(test_labels.shape, float(development_majority)),
    ),
}
display(pd.DataFrame([
    {'model': name, **{key: value for key, value in result.items() if key != 'per_class'}}
    for name, result in diagnostic_results.items()
]))
per_class_frame = pd.DataFrame([
    {'model': name, **row}
    for name, result in diagnostic_results.items() for row in result['per_class']
])
display(per_class_frame)
px.bar(per_class_frame.assign(class_label=lambda frame: 'Class ' + frame['class'].astype(str)),
       x='class_label', y='recall', color='model', barmode='group', range_y=[0, 1.05],
       hover_data=['support', 'precision', 'f1'],
       title='Recall by class: does high accuracy hide an ignored class?').show()

probability_frame = pd.DataFrame({
    'actual_class': ['Class ' + str(int(label)) for label in test_labels],
    'probability_class_1': test_probabilities,
})
display(probability_frame.groupby('actual_class')['probability_class_1']
        .agg(['count', 'min', 'max']).style.format({'min': '{:.12g}', 'max': '{:.12g}'}))
px.histogram(probability_frame, x='probability_class_1', color='actual_class',
             nbins=50, barmode='overlay', opacity=0.65, range_x=[0, 1],
             title='Held-out confidence: separation is not the same as class balance').show()
largest_negative = float(np.max(test_probabilities[test_labels == 0]))
smallest_positive = float(np.min(test_probabilities[test_labels == 1]))
if largest_negative < smallest_positive:
    print(f'Every threshold t with {largest_negative:.12g} < t <= '
          f'{smallest_positive:.12g} classifies this sample correctly (prediction uses p >= t).')
else:
    print('The class probability ranges overlap: no single threshold classifies every test row correctly.')

,split,samples,class_0,class_1,class_1_fraction,majority_minority_ratio
0,Unique data,1348,738,610,0.452522,1.209836
1,Development,1078,590,488,0.452690,1.209016
2,Locked test (descriptive only),270,148,122,0.451852,1.213115
3,Fold 1 training,718,393,325,0.452646,1.209231
4,Fold 1 validation,360,197,163,0.452778,1.208589
5,Fold 2 training,718,393,325,0.452646,1.209231
6,Fold 2 validation,360,197,163,0.452778,1.208589
7,Fold 3 training,720,394,326,0.452778,1.208589
8,Fold 3 validation,358,196,162,0.452514,1.209877


,model,accuracy,balanced_accuracy,macro_f1,brier_score
0,Selected ANN (threshold 0.5),1.000000,1.0,1.000000,3.516456e-11
1,Development-majority baseline,0.548148,0.5,0.354067,4.518519e-01


,model,class,support,precision,recall,f1
0,Selected ANN (threshold 0.5),0,148,1.000000,1.0,1.000000
1,Selected ANN (threshold 0.5),1,122,1.000000,1.0,1.000000
2,Development-majority baseline,0,148,0.548148,1.0,0.708134
3,Development-majority baseline,1,122,0.000000,0.0,0.000000


,count,min,max
actual_class,,,
Class 0,148,7.11283875344e-20,7.52201499621e-05
Class 1,122,0.999970112234,1


Every threshold t with 7.52201499621e-05 < t <= 0.999970112234 classifies this sample correctly (prediction uses p >= t).


:) so the above graphs so that the high accuracy is not merely an effect of predicting the majority class: the model correctly identifies both classes on this test set.

## Going further: nested cross-validation

A perfect score on one split is not enough. This study uses **five outer folds on the 1,078-row development set only**, keeping the existing 270-row holdout out of the study entirely. Every development row gets exactly one out-of-fold prediction from a model that did not train on it.

For **each outer fold**, repeat the original selection procedure from scratch: screen all 14 architectures, refine the top three using all three PSO profiles, and rank the 23 candidates by inner validation accuracy, then BCE, then parameter count. Inner selection uses three stratified folds and two optimizer seeds per fold (11 and 29, with the notebook's fold offsets). All scalers are fitted within their respective training partition. Refit the winning configuration on outer-training rows using seeds 101/202/303 and choose the restart by training objective only. Evaluate it once on the outer-validation rows at the fixed 0.5 threshold. No outer scores choose configurations or restarts.

In [41]:
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display


data_path = next(
    (root / 'outline' / 'data_banknote_authentication.txt'
     for root in (Path.cwd(), *Path.cwd().parents)
     if (root / 'outline' / 'data_banknote_authentication.txt').is_file()),
    None,
)
if data_path is None:
    raise FileNotFoundError('Could not locate outline/data_banknote_authentication.txt from the current directory.')
nested_directory = data_path.parent.parent / 'Riddhi' / 'ann_results' / 'nested_cv'
nested_path = nested_directory / 'results.json'
if not nested_path.is_file():
    raise FileNotFoundError('Run python -m src.evaluation.riddhi_nested_cv from the project root first.')
nested_results = json.loads(nested_path.read_text())
if nested_results['data_sha256'] != hashlib.sha256(data_path.read_bytes()).hexdigest():
    raise ValueError('Nested CV results belong to a different source dataset.')
nested_predictions = pd.read_csv(nested_directory / 'out_of_fold_predictions.csv')
if (nested_predictions['original_row_index'].duplicated().any()
        or set(nested_predictions['original_row_index']) != set(nested_results['development_original_row_indices'])
        or set(nested_predictions['original_row_index']) & set(nested_results['excluded_holdout_original_row_indices'])):
    raise ValueError('Out-of-fold rows do not match the development-only protocol.')
nested_fold_frame = pd.DataFrame([
    {
        'Outer fold': fold['fold'], 'Validation rows': fold['report']['samples'],
        'Architecture': ' -> '.join(map(str, [4, *fold['selected']['config']['hidden'], 1])),
        'Activations': ' / '.join(fold['selected']['config']['activations']),
        'Inner-selected accuracy': fold['selected']['mean_accuracy'],
        'Outer accuracy': fold['report']['accuracy'],
        'Outer balanced accuracy': fold['report']['balanced_accuracy'],
        'Outer F1 (class 1)': fold['report']['f1_class_1'],
        'Outer BCE': fold['report']['binary_cross_entropy'],
        'Majority baseline': fold['report']['majority_baseline_accuracy'],
        'Errors': fold['report']['confusion_matrix'][0][1] + fold['report']['confusion_matrix'][1][0],
    }
    for fold in nested_results['folds']
])
display(nested_fold_frame)
print(f"Mean outer accuracy: {nested_results['mean_outer_accuracy']:.4%}; "
      f"fold SD: {nested_results['std_outer_accuracy']:.4%} (not a confidence interval).")
pooled = nested_results['pooled_out_of_fold']
errors = int(np.sum(nested_predictions['actual_class'] != nested_predictions['predicted_class']))
print(f"Out-of-fold accuracy: {pooled['accuracy']:.4%}; "
      f"errors: {errors}/{len(nested_predictions)}; "
      f"balanced accuracy: {pooled['balanced_accuracy']:.4%}; "
      f"F1 (class 1): {pooled['f1_class_1']:.4%}; AUC: {pooled['roc_auc']:.6f}.")
plot_data = nested_fold_frame.melt(
    id_vars='Outer fold', value_vars=['Outer accuracy', 'Outer balanced accuracy', 'Majority baseline'],
    var_name='Metric', value_name='Score',
)
figure = px.bar(plot_data, x='Outer fold', y='Score', color='Metric', barmode='group',
                 range_y=[0, 1.05], title='Nested CV: genuinely outer-held-out performance in each fold')
figure.update_yaxes(tickformat='.0%')
figure.show()
matrix = np.asarray(pooled['confusion_matrix'])
px.imshow(matrix, x=['Predicted 0', 'Predicted 1'], y=['Actual 0', 'Actual 1'],
          text_auto=True, color_continuous_scale='Blues',
          title='Pooled out-of-fold confusion matrix (five separately selected models)').show()
print('Out-of-fold mistakes (empty means no mistakes in this study):')
display(nested_predictions.loc[nested_predictions['actual_class'] != nested_predictions['predicted_class']])
print(nested_results['interpretation'])

,Outer fold,Validation rows,Architecture,Activations,Inner-selected accuracy,Outer accuracy,Outer balanced accuracy,Outer F1 (class 1),Outer BCE,Majority baseline,Errors
0,1,216,4 -> 12 -> 1,tanh / logistic,1.000000,0.99537,0.995763,0.994924,3.645671e-03,0.546296,1
1,2,216,4 -> 8 -> 4 -> 1,relu / relu / logistic,1.000000,1.00000,1.000000,1.000000,1.786386e-05,0.546296,0
2,3,216,4 -> 12 -> 1,relu / logistic,1.000000,1.00000,1.000000,1.000000,5.282274e-06,0.546296,0
3,4,215,4 -> 8 -> 4 -> 1,relu / tanh / logistic,0.999419,1.00000,1.000000,1.000000,9.507781e-07,0.548837,0
4,5,215,4 -> 12 -> 1,relu / logistic,1.000000,1.00000,1.000000,1.000000,3.218572e-04,0.548837,0


Mean outer accuracy: 99.9074%; fold SD: 0.2070% (not a confidence interval).
Out-of-fold accuracy: 99.9072%; errors: 1/1078; balanced accuracy: 99.9153%; F1 (class 1): 99.8976%; AUC: 1.000000.


Out-of-fold mistakes (empty means no mistakes in this study):


,fold,development_index,original_row_index,actual_class,probability_class_1,predicted_class
1007,1,1007,349,0,0.544271,1


Fold SD is descriptive, not a confidence interval. Pooled Wilson interval assumes independent observations; folds share training data. Previously used dataset/search space and unknown specimen dependence limit external generalisation claims.
